# 02. PEC Metrics, Statistical Checks, and Diversity Audit

This notebook computes the core measurement quantities behind RQ1-RQ3.
It also shows why the paper treats entropy carefully: raw exposure
diversity is high, but count matching is needed because recommendation
lists contain many more items than click histories.

This is the main notebook for understanding why the paper separates raw distribution gaps from count-matched audit signals.


<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:statistics_guide -->

## Reader guide: statistical logic in this notebook

This notebook computes the paper's core measurement statistics for RQ1-RQ3. The main theme is **measurement validity**: raw values can be misleading if exposure and consumption are observed at different scales.

Important checks include:

- **Bootstrap confidence intervals:** repeated resampling of users to estimate uncertainty around means.
- **Date-matched random baselines:** random comparisons restricted to the same date so that news availability and daily news cycles are not ignored.
- **Popularity-aware baselines:** checks whether recommendation traceability is merely reflecting generally popular items.
- **Count-matched diversity:** compares exposure and clicks after matching the number of items, reducing the sparse-click artifact.

These checks are designed to make the audit conservative rather than to maximize headline numbers.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:diversity_methods -->

## Diversity and concentration methods

The paper deliberately uses multiple diversity/concentration metrics because each captures a different failure mode.

- **Entropy** increases when categories or publishers are more evenly distributed.
- **HHI** increases when the distribution is concentrated in a few labels.
- **Top-share** captures dominance by the single most frequent label.
- **Effective number of categories** translates concentration into an intuitive count-like scale.
- **Jensen-Shannon divergence** compares the full exposure distribution against the full click distribution for the same user.

A key result is that raw entropy gaps shrink after count matching, but concentration and distributional mismatch remain. That is why the paper avoids claiming a simple algorithmic filter-bubble effect and instead argues for a bounded diversity-shift audit.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "02_pec_metrics_and_statistics"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PROCESSED.exists()}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PROCESSED.exists():
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`."
    ))


In [ ]:
def safe_parse(value):
    '''Parse JSON-like strings stored in CSV cells.''';
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            continue
    return None


def truthy(series: pd.Series) -> pd.Series:
    '''Normalize common true-like CSV values to booleans.''';
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def pref_set(row: pd.Series, primary: str, fallback: str) -> set[str]:
    '''Return a stated preference set from explicit selections or weighted profile state.

    primary is usually the user-selection column. fallback is the current weighted
    profile-state column. For a weighted dictionary, small positive weights are
    treated as selected dimensions for set-overlap analyses.
    '''
    parsed = safe_parse(row.get(primary, ""))
    if parsed is None:
        parsed = safe_parse(row.get(fallback, ""))
    if isinstance(parsed, dict):
        out = set()
        for key, value in parsed.items():
            try:
                if float(value) > 0.01:
                    out.add(str(key))
            except Exception:
                out.add(str(key))
        return out or set(map(str, parsed.keys()))
    if isinstance(parsed, list):
        out = set()
        for item in parsed:
            if isinstance(item, dict) and "value" in item:
                out.add(str(item["value"]))
            else:
                out.add(str(item))
        return out
    return set()


def normalize_dist(values: dict[str, float]) -> dict[str, float]:
    clean = {}
    for key, value in values.items():
        try:
            val = float(value)
        except Exception:
            continue
        if key and val > 0:
            clean[str(key)] = val
    total = sum(clean.values())
    if total <= 0:
        return {}
    return {key: val / total for key, val in clean.items()}


def parse_weight_dist(value) -> dict[str, float]:
    '''Parse a weighted preference dictionary such as {'Politics': 0.7, ...}.''';
    parsed = safe_parse(value)
    if isinstance(parsed, dict):
        return normalize_dist({str(k): v for k, v in parsed.items()})
    if isinstance(parsed, list):
        vals = []
        for item in parsed:
            if isinstance(item, dict) and "value" in item:
                vals.append(str(item["value"]))
            else:
                vals.append(str(item))
        if vals:
            return {key: 1 / len(vals) for key in vals if key}
    return {}


def norm_entropy(values) -> float:
    '''Normalized entropy in [0, 1]. Higher means more diverse.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    counts = values.value_counts().to_numpy(dtype=float)
    probs = counts / counts.sum()
    ent = -(probs * np.log(probs)).sum()
    max_ent = np.log(len(counts)) if len(counts) > 1 else 1.0
    return float(ent / max_ent) if max_ent else 0.0


def norm_entropy_dist(dist: dict[str, float]) -> float:
    values = np.array([v for v in dist.values() if v > 0], dtype=float)
    if len(values) == 0:
        return 0.0
    values = values / values.sum()
    ent = float(-(values * np.log(values)).sum())
    denom = math.log(len(values)) if len(values) > 1 else 1.0
    return ent / denom if denom else 0.0


def top_share(values) -> float:
    '''Share of the most common item. Higher means more concentrated.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    return float(values.value_counts().iloc[0] / len(values))


def hhi(values) -> float:
    '''Herfindahl-Hirschman concentration index. Higher means more concentrated.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    probs = values.value_counts(normalize=True).to_numpy(dtype=float)
    return float(np.sum(probs * probs))


def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return np.nan
    return len(a & b) / len(a | b)


def distribution_from_values(values) -> dict[str, float]:
    series = pd.Series(values).dropna().astype(str)
    series = series[series != ""]
    if series.empty:
        return {}
    counts = series.value_counts()
    return {str(k): float(v) / float(counts.sum()) for k, v in counts.items()}


def cosine(a: dict[str, float], b: dict[str, float]) -> float:
    keys = sorted(set(a) | set(b))
    if not keys:
        return np.nan
    av = np.array([a.get(k, 0.0) for k in keys], dtype=float)
    bv = np.array([b.get(k, 0.0) for k in keys], dtype=float)
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom > 0 else np.nan


def js_divergence(a: dict[str, float], b: dict[str, float]) -> float:
    '''Jensen-Shannon divergence between two discrete distributions.''';
    keys = sorted(set(a) | set(b))
    if not keys:
        return np.nan
    p = np.array([a.get(k, 0.0) for k in keys], dtype=float)
    q = np.array([b.get(k, 0.0) for k in keys], dtype=float)
    if p.sum() <= 0 or q.sum() <= 0:
        return np.nan
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    def kl(x, y):
        mask = x > 0
        return float((x[mask] * np.log2(x[mask] / y[mask])).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def dist_top_key(dist: dict[str, float]) -> str:
    return max(dist.items(), key=lambda item: item[1])[0] if dist else ""


def classify_surface(row: pd.Series) -> str:
    '''Map raw app parameters to the surface categories used in the paper.''';
    source = str(row.get("p_source", "") or "").lower()
    section = str(row.get("section", "") or "").lower()
    message = str(row.get("message_type", "") or "").lower()
    is_headline = str(row.get("is_headline", "") or "").lower() in {"1.0", "1", "true", "yes"}
    if "push" in message or "notification" in source or "notification" in section:
        return "notification"
    if is_headline or "headline" in section:
        return "headline"
    if section.startswith("category_"):
        return "category"
    if "for_you" in section or source == "home":
        return "home/feed"
    if "related" in section or source == "news_detail":
        return "article-detail"
    if "publisher" in section or source in {"newsroom", "source"}:
        return "newsroom"
    if "podcast" in section or "podcast" in source:
        return "podcast"
    if "insight" in section or "insight" in source:
        return "insights"
    if "search" in section or source == "search":
        return "search"
    return "other/unknown"


def mask_id(value: str) -> str:
    '''Mask user identifiers before showing examples in a notebook.''';
    text = str(value)
    if len(text) <= 8:
        return text[:2] + "***"
    return text[:4] + "***" + text[-4:]


In [ ]:
profiles = pd.read_csv(PROCESSED / "user_profiles_clean.csv", dtype=str).fillna("")
clicks = pd.read_csv(PROCESSED / "click_events_enriched.csv", dtype=str).fillna("")
recs = pd.read_csv(PROCESSED / "recommendation_items_enriched.csv", dtype=str).fillna("")
matched_clicks = clicks.loc[truthy(clicks["news_matched"])].copy()
matched_recs = recs.loc[truthy(recs["news_matched"])].copy()


In [ ]:
# Load saved tables as a reference, then recompute the key diversity fields
# directly below. This keeps the notebook useful both for learning and checking.
saved_rq1 = pd.read_csv(TABLES / "rq1_preference_consumption_gap_by_user.csv")
saved_rq2 = pd.read_csv(TABLES / "rq2_click_exposure_rows.csv")
saved_rq3 = pd.read_csv(TABLES / "rq3_diversity_by_user.csv")

display(pd.DataFrame({
    "table": ["rq1", "rq2_click_rows", "rq3"],
    "rows": [len(saved_rq1), len(saved_rq2), len(saved_rq3)],
}))


In [ ]:
# Bootstrap helper for user-level uncertainty intervals.
def bootstrap_mean_ci(values, n_boot=2000, seed=42):
    arr = pd.Series(values).dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.array([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)])
    return float(arr.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

metrics = []
for name, series in [
    ("RQ1 category Jaccard top-3", saved_rq1["category_jaccard_top3"]),
    ("RQ1 click category entropy", saved_rq1["click_category_entropy"]),
    ("RQ2 same-day hit among comparable clicks", saved_rq2.loc[saved_rq2["has_same_day_recommendations"].astype(bool), "clicked_in_same_day_recs"].astype(float)),
    ("RQ3 rec category entropy", saved_rq3["rec_category_entropy"]),
    ("RQ3 click category entropy", saved_rq3["click_category_entropy"]),
    ("RQ3 rec top-category share", saved_rq3["rec_top_category_share"]),
    ("RQ3 click top-category share", saved_rq3["click_top_category_share"]),
]:
    mean, lo, hi = bootstrap_mean_ci(series)
    metrics.append({"metric": name, "mean": mean, "ci95_low": lo, "ci95_high": hi})
display(pd.DataFrame(metrics))


In [ ]:
# Recompute diversity by user from the processed click and recommendation rows.
click_groups = matched_clicks.groupby("user_id")
rec_groups = matched_recs.groupby("user_id")
users = sorted(set(click_groups.groups) | set(rec_groups.groups))

rows = []
for user_id in users:
    c = click_groups.get_group(user_id) if user_id in click_groups.groups else pd.DataFrame()
    r = rec_groups.get_group(user_id) if user_id in rec_groups.groups else pd.DataFrame()

    click_cats = c.get("category_final", pd.Series(dtype=str)).replace("", pd.NA).dropna().astype(str).tolist()
    rec_cats = r.get("category", pd.Series(dtype=str)).replace("", pd.NA).dropna().astype(str).tolist()
    click_pubs = c.get("publisher_id_final", pd.Series(dtype=str)).replace("", pd.NA).dropna().astype(str).tolist()
    rec_pubs = r.get("publisher_id", pd.Series(dtype=str)).replace("", pd.NA).dropna().astype(str).tolist()

    rows.append({
        "user_id": user_id,
        "click_count": len(click_cats),
        "rec_count": len(rec_cats),
        "click_category_entropy": norm_entropy(click_cats),
        "rec_category_entropy": norm_entropy(rec_cats),
        "click_publisher_entropy": norm_entropy(click_pubs),
        "rec_publisher_entropy": norm_entropy(rec_pubs),
        "click_top_category_share": top_share(click_cats),
        "rec_top_category_share": top_share(rec_cats),
        "click_category_hhi": hhi(click_cats),
        "rec_category_hhi": hhi(rec_cats),
    })
diversity = pd.DataFrame(rows)
display(diversity.describe().T)


In [ ]:
# Raw diversity and concentration comparison.
raw = pd.DataFrame([
    {"metric": "Category entropy", "Exposure": diversity["rec_category_entropy"].mean(), "Consumption": diversity["click_category_entropy"].mean()},
    {"metric": "Publisher entropy", "Exposure": diversity["rec_publisher_entropy"].mean(), "Consumption": diversity["click_publisher_entropy"].mean()},
    {"metric": "Top-category share", "Exposure": diversity["rec_top_category_share"].mean(), "Consumption": diversity["click_top_category_share"].mean()},
    {"metric": "Category HHI", "Exposure": diversity["rec_category_hhi"].mean(), "Consumption": diversity["click_category_hhi"].mean()},
])
display(raw)

ax = raw.set_index("metric")[["Exposure", "Consumption"]].plot(kind="bar", figsize=(10, 4), color=["#2563EB", "#DC2626"])
ax.set_title("Raw exposure-consumption metrics")
ax.set_ylabel("Metric value")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Count-matched audit: for each user, sample the same number of exposure
# items as clicks and recompute the exposure entropy/concentration.
rng = np.random.default_rng(44)
n_iter = 300
cm_rows = []

for user_id in sorted(set(click_groups.groups) & set(rec_groups.groups)):
    c = click_groups.get_group(user_id)
    r = rec_groups.get_group(user_id)
    click_cats = c["category_final"].replace("", pd.NA).dropna().astype(str).tolist()
    rec_cats = r["category"].replace("", pd.NA).dropna().astype(str).tolist()
    n_click = len(click_cats)
    if n_click < 5 or len(rec_cats) < n_click:
        continue
    arr = np.array(rec_cats, dtype=object)
    sampled_entropy = []
    sampled_hhi = []
    sampled_top_share = []
    for _ in range(n_iter):
        sample = rng.choice(arr, size=n_click, replace=False).tolist()
        sampled_entropy.append(norm_entropy(sample))
        sampled_hhi.append(hhi(sample))
        sampled_top_share.append(top_share(sample))
    cm_rows.append({
        "user_id": user_id,
        "click_count": n_click,
        "rec_count": len(rec_cats),
        "click_entropy": norm_entropy(click_cats),
        "count_matched_rec_entropy": float(np.nanmean(sampled_entropy)),
        "entropy_gap_count_matched": float(np.nanmean(sampled_entropy) - norm_entropy(click_cats)),
        "click_hhi": hhi(click_cats),
        "count_matched_rec_hhi": float(np.nanmean(sampled_hhi)),
        "hhi_gap_count_matched": float(hhi(click_cats) - np.nanmean(sampled_hhi)),
        "click_top_share": top_share(click_cats),
        "count_matched_rec_top_share": float(np.nanmean(sampled_top_share)),
        "top_share_gap_count_matched": float(top_share(click_cats) - np.nanmean(sampled_top_share)),
    })

count_matched = pd.DataFrame(cm_rows)
display(count_matched.describe().T)


In [ ]:
# Interpretation table: entropy and concentration move in opposite directions.
summary_rows = []
for col in ["entropy_gap_count_matched", "hhi_gap_count_matched", "top_share_gap_count_matched"]:
    mean, lo, hi = bootstrap_mean_ci(count_matched[col])
    summary_rows.append({"metric": col, "mean": mean, "ci95_low": lo, "ci95_high": hi})
display(pd.DataFrame(summary_rows))


In [ ]:
# Date-matched random exposure baseline, shown as a robustness diagnostic.
# This is NOT a popularity baseline. It preserves the date and list-size setting.
topk_path = TABLES / "optimization_topk_exposure.csv"
if topk_path.exists():
    topk = pd.read_csv(topk_path)
    display(topk[["top_k", "same_day_hit_rate", "random_hit_rate_mean", "lift", "random_ci_95_low", "random_ci_95_high"]])
else:
    print("optimization_topk_exposure.csv not found; run scripts/run_icwsm_optimization_checks.py to regenerate it.")


**Why this matters for the paper**

- Entropy is a diversity metric: higher means broader distribution.
- Top-share and HHI are concentration metrics: higher means more focused distribution.
- Raw entropy gaps are large partly because logged recommendation lists are longer than click histories.
- Count matching reduces the entropy gap, but concentration/distributional mismatch remains.
- That is the careful version of the RQ3 claim.


## Metric map: why there are several diversity statistics

This notebook does not reduce diversity to a single number because each statistic captures a different property.

- **Entropy:** how evenly a distribution is spread. It can increase mechanically when more items are observed, which motivates count matching.
- **HHI:** concentration in a small number of labels. Larger values indicate greater concentration.
- **Top-share:** the proportion held by the most frequent label. It gives an intuitive view of dominant-category concentration.
- **JS divergence:** the difference in overall shape between exposure and click distributions.

RQ3 therefore does not end with the claim that raw exposure is more diverse than clicks. The more careful question is whether HHI, top-share, and JS mismatch remain after count matching reduces the entropy gap.


In [ ]:
metric_map = pd.DataFrame([
    {
        "metric": "Normalized Shannon entropy",
        "question it answers": "labels are spread evenly or narrowly?",
        "higher means": "more even / more diverse distribution",
        "paper use": "raw and count-matched exposure-vs-click breadth comparison",
        "main caution": "can be inflated when one side has many more observed items",
    },
    {
        "metric": "HHI concentration",
        "question it answers": "is mass concentrated in few labels?",
        "higher means": "more concentrated / less diverse distribution",
        "paper use": "count-adjusted concentration mismatch after entropy gap shrinks",
        "main caution": "direction is opposite of entropy; higher is narrower",
    },
    {
        "metric": "Top-share",
        "question it answers": "how dominant is the single largest label?",
        "higher means": "one category/source dominates consumption or exposure",
        "paper use": "shows whether clicks concentrate around one top category more than exposure",
        "main caution": "only sees the top label, not the full distribution shape",
    },
    {
        "metric": "Jensen-Shannon divergence",
        "question it answers": "are the exposure and click distributions shaped similarly?",
        "higher means": "larger distributional mismatch",
        "paper use": "full-distribution mismatch between logged exposure and click consumption",
        "main caution": "does not say which side caused the mismatch",
    },
    {
        "metric": "Count matching",
        "question it answers": "does the gap remain when exposure is sampled down to click count?",
        "higher means": "not a metric itself; it is a control procedure",
        "paper use": "separates count artifact from substantive distribution mismatch",
        "main caution": "answers a bounded comparison, not causal effect",
    },
])
display(metric_map)


## Reading the RQ3 result without overclaiming

Raw recommendation lists contain many more items than click histories. If exposure has 100 observed items and clicks have 5 observed items, raw exposure entropy can look much larger simply because there were more chances to observe categories. That is the **observation-count artifact**.

Count matching asks: “If I sample the exposure side down to the user's click count, does the exposure distribution still look broader or less concentrated?” In the final paper, entropy gap becomes much smaller after this control. That is honest and important. The stronger remaining result is that concentration and full-distribution mismatch, especially HHI/top-share/JS, still remain in the audit-eligible cohort.

So the correct RQ3 sentence is not “the recommender diversifies attention.” It is: logged exposure and click consumption remain distributionally different under count-adjusted concentration and JS diagnostics.

In [ ]:
# AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
# This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
# It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
_save_names = ['metrics', 'diversity', 'raw', 'count_matched', 'summary_rows', 'topk', 'metric_map']
_saved_paths = []
for _name in _save_names:
    if _name not in globals():
        continue
    _obj = globals()[_name]
    _path = SCRATCH_TABLES / f"{_name}.csv"
    try:
        if isinstance(_obj, pd.DataFrame):
            _obj.to_csv(_path, index=False)
        elif isinstance(_obj, pd.Series):
            _obj.to_frame().to_csv(_path)
        elif isinstance(_obj, list):
            pd.DataFrame(_obj).to_csv(_path, index=False)
        elif isinstance(_obj, dict):
            pd.DataFrame([_obj]).to_csv(_path, index=False)
        else:
            continue
        _saved_paths.append(_path)
    except Exception as exc:
        print(f"Could not save {_name}: {exc}")

try:
    import matplotlib.pyplot as _plt
    for _idx, _num in enumerate(_plt.get_fignums(), 1):
        _fig = _plt.figure(_num)
        _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
        _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
        _saved_paths.append(_fig_path)
except Exception as exc:
    print(f"Could not save open matplotlib figures: {exc}")

print("Saved scratch artifacts:")
for _path in _saved_paths:
    print(" -", _path)
